# Week 15 - PDF Parsing

When designing the course I keep asking myself "What has been personally useful to me as a programmer?"

One such topic which came to mind is PDF Parsing.

There are many different open-source community built libraries for PDF parsing.

I recently made a program to parse data from applicants to our Master's Program. To do so I tried a few libraries, and found the one that worked for me was called "pdfminer". The most recent release can be installed through the library name "pdfminer-six".

So let's install it and get started. Along the way we'll also touch on the following new topics:

* Exploring folders and files from python
* Try Except blocks
* Decorators

In [13]:
%pip install pdfminer-six

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### A Very Simple Example

Parsing the text from an admissions questionnaire.

In [ ]:
from pdfminer.high_level import extract_text

In [3]:
file = './questionnaire.pdf'

text = extract_text(file)

In [4]:
text

'DIGITAL CULTURAL HER ITAGE  STUDIES  \n\nQuestionnaire for the Admission Procedure \nM. A. Digital Cultural Heritage\n\nI Personal Data \n\nAddress / Surname: \n\nFirst name(s): \n\nDate of Birth: \n\nPlace/Country of Birth: \n\nNationality: \n\nCurrent Home Address: \n\nE-mail Address: \n\nTelephone: \n\nII Details on Previous Studies \n\nPlease  provide  us  with  the  full  details  on  all  previous  university  degrees,  that  you  have \nearned so far, or that you are going to complete before the start date of the M. A. program \nDigital Cultural Heritage in October of the current year. In case of more than one program, \nuse the second and third column respectively. \n\n1 \n\n2 \n\n3 \n\nDegree: \n(B. A., B. Sc., M.A., \nM. Sc. etc.)\nFull Name of \nProgram: \n(and minors if \napplicable) \n\nFull name of \nUniversity: \n\nYear of \ngraduation \n\n\x0cLUDWIG-MAXIMILIANS-UNI VERSITÄT MÜNCHEN  \n\nSEITE 2 VON  2 \n\nIII Agreement on Notification by E-mail \n\nIf  the  admission  

But wait, the filled-in fields are not included. How do we get those?

The data structure for a PDF is quite complicated. It is stored in a complex dictionary which includes many data types (and even some nested dictonaries!).

The filled in data is under the "AcroForm" key, and within this the individual entries are stored under the "fields" key.

In [ ]:
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfdocument import PDFDocument
from pdfminer.psparser import PSLiteral
from pdfminer.pdftypes import resolve1, PDFObjRef

file = './questionnaire.pdf'

data = {}

with open(file, 'rb') as fp:
    parser = PDFParser(fp)
    doc = PDFDocument(parser)

    # The "resolve1" function changes the pdf datatypes to a normal dictionary python can handle
    fields = resolve1(doc.catalog['AcroForm'])['Fields']
    for f in fields:
        # For each filled-in field in the form the following two lines return the name of the field (name), and the entered text (value)
        field = resolve1(f)
        name, value = field.get('T'), field.get('V')

        # Unfortunately there can be instances the name and value are not normal strings
        # In this case the following 3 "if statements" are necessary to deal with that
        if isinstance(name, PSLiteral):
            name = name.name
        if isinstance(value, PDFObjRef):
            value = resolve1(value)
        if isinstance(value, PSLiteral):
            value = value.name
        
        # The field values are in binary. We need to change that to standard "utf-8" encoding to view them
        if value is not None and not isinstance(value,(str,dict)):
            value = value.decode('utf-8')
            
        # And now we can save the decoded information in a key-value pair in our data dictionary
        data[name.decode('utf-8')] = value

In [9]:
data

{'Surname': 'Eames',
 'First name(s)': 'Evan',
 'Date of Birth': '22/07/1990',
 'Place of Birth': 'Canada',
 'Nationality': 'Canadian',
 'Current Home Address': 'Holzstr. 37, 80469, Munich, DE',
 'E-mail Address': 'evan.eames@lmu.de',
 'Telephone': '01783717245',
 '1st Degree': 'BSc',
 '2nd Degree': 'MSc',
 '3rd Degree': 'PhD',
 '2 Full Name of Program': 'Astrophysics',
 '1 Full Name of Program': 'Honours Physics',
 '3 Full Name of Program': 'Astrophysics',
 '1 University': 'McGill University',
 '2 University': 'University of Manchester',
 '3 University': 'Paris Sciences et Lettres',
 '1 Year of graduation': '2014',
 '2 Year of graduation': '2015',
 '3 Year of graduation': '2018',
 'E-mail-Notification': 'Yes',
 'Place name': 'Munich',
 'Date of signature': '10/05/2026',
 'Signature': None,
 'Form of address': 'Mr.'}

In [ ]:
# Just for fun, we can look at the rest of the PDF data structure.

doc.catalog

{'AcroForm': <PDFObjRef:158>,
 'Lang': b'de-DE',
 'MarkInfo': {'Marked': True},
 'Metadata': <PDFObjRef:14>,
 'Pages': <PDFObjRef:128>,
 'StructTreeRoot': <PDFObjRef:34>,
 'Type': /'Catalog',
 'ViewerPreferences': <PDFObjRef:159>}

# Some more stuff

### Checking what's in a directory

In [11]:
import os

directory_path = '../../../Admissions_2026'

dir_names = []

with os.scandir(directory_path) as entries:
    for entry in entries:
        if entry.is_dir():
            dir_names.append(entry.name)

print(dir_names)


['001_Yuhan_XIA', '002_Elizabeth_LONGO', '003_Sute_IWAR', '004_Crescenzo_DIGIANNANTONIO', '005_Liu_ZIJIA', '006_Matthew_CURRIE', '007_Ezinne_Favour_NDIONYEMA', '008_Mohammad_Shamiur_RAHMAN', '009_Dennis_UMEAKUBILO', '010_Jacinta_ATAYDE', '011_Yanrong_ZHU', '012_Muhammed_Mansoor_GUJJAR', '013_Saba_TARIQ', '014_Zhen_QIAN', '015_Manlin_MA']


### Checking which files are in a directory

In [ ]:
for dir in dir_names:
    print(dir)
    applicant = initializeApplicant(dir)
    with os.scandir(directory_path + '/' + dir) as entries:
        for entry in entries:
            if entry.is_file():
                if 'questionnaire' in entry.name.lower():
                    applicant.questionnaire = True
                    questionnaire_file = entry.name
                    applicant_dir = directory_path + '/' + dir + '/' + questionnaire_file
                    extractApplicantInfo(applicant_dir, applicant)

### Try-Except Blocks

This handy trick will keep the code running even if there are errors.

Very useful alongside PDF parsing, where you can't really predict everything that could go wrong with a file structure as complex ast the PDF (example: digital signatures tend to break things when parsing)

In [20]:
# This breaks stuff

1/0

print("Continuing...")

ZeroDivisionError: division by zero

In [ ]:
# This doesn't

try:
    1/0
    
except:
    print("You can't divide by zero!")

print("Continuing...")

You can't divide by zero!
Continuing...


### Decorators

A "decorator" is a library that is added in front of a function or class and "enhances" the behaviour.

In python you apply a decorator with the at symbol (@).

The below example shows the "dataclass" decorator. It is a helpful decorator for when you have a class with a lot of attributes.

What does it do? It saves you the trouble of having to explicitly define an __init__ function.

Here's the example:

In [ ]:
# Without decorator = Very inconvenient:

class Applicant:
    def __init_(self, id:int, first_name:str, last_name:str, date_of_birth=None, place_of_birth=None, degrees=[], questionnaire=False):
        self.id = id
        self.first_name = first_name
        self.last_name = last_name
        self.date_of_birth = date_of_birth
        self.place_of_birth = place_of_birth
        self.degrees=[]
        self.questionnaire=False

In [ ]:
# With decorator = Much more concise!

from dataclasses import dataclass, field

@dataclass  # <-- This is the decorator
class Applicant:
    id: int
    first_name: str
    last_name: str
    date_of_birth=None
    place_of_birth=None
    degrees: list = field(default_factory=list)  # Note: This line is necessary to create a new empty array each time an object is initialized
    questionnaire=False

# Note: You can expand on this with more attributes in the subsequent project

# Project

### System to Process Applications

This will be a larger project that needs to be sent to me by June 21st, 23:59.

It can add up to 10% to the final grade.

You are a program coordinator and are trying to make a program that can parse through application documents and output two csv file:

* The first file has all the relevant information of the applicants, with one applicant on each row. This will be extracted from the "questionnaire" file (if it exists).

* The second file is a checklist file. Each row again represents an applicant, but instead of listing information about the applicant, the subsequent columns are about the provided documents. There are 3 documents that should be provided (questionnaire, transcript or degree or diploma, and cv). An applicant with all 3 will result in a row that looks like:

"First Name", "Last Name", True, True, True

If the CV is missing, the row will instead be:

"First Name", "Last Name", True, True, False

**Note**: Each student should send their files to me individually as a zip file. The zip file should include:

* A file called "main.py" which contains the main code (note it should NOT be a .ipynb file!)

* A file called "utils.py" which should contain any function definitions

* A file "applicant_class.py" which contains an "Applicant" class

The file "main.py" should import the functions like:

from utils import function1, function2, etc...

and it should also import the class like:

from applicant_class import Applicant

**Note**: Students will be asked to walk us through their code, so although you can work together, it's not worth copying or using an LLM. You really need to understand each line of code!

### How to get started:

There is a zip file on Moodle. It contains a directory (folder) of applications with sub-directories (sub-folders), one for each applicant. Download it and unzip it. The directory should be in the same place as your main.py, utils.py, and applicant_class.py files.

There are many different ways you could approach this project, but here are some hints that could help.

1) Initialize empty pandas dataframes, one for the info csv that will be output, and the other for the checklist csv that will be output.

2) You will want to scan the file system to find the names of all the sub-directories. From these you can make a list of all the applicant folders.

3) Now loop through them. For each one initialize an Applicant object. You can already assign the id, first name, and last name from the directory name, which has the form (id_firstname_lastname).

4) While still in the loop, also look at all the files present in the directory of the current applicant and update the object's attributes accordingly.

5) If the object's "questionnaire" attribute is True, then create a function that can parse all of the relevant info from the questionnaire file and update the object's attributes accordingly.

6) Inside the Applicant class, add 2 functions. One should output a dictionary representing all of the applicant info, the other should output a dictionary representing the files for the applicant {"first name":john, "second name: "smith", "questionnaire":True, ...}.

7) Before moving on to the next applicant, append the dictionary to the end of the DataFrame as a new row. The code below is how to do this:

In [ ]:
import pandas as pd

if len(info_df) == 0:  # If the dataframe is empty we need to initialize it with the first row
    info_df = pd.DataFrame(applicant.app_info_to_dict(), index=[0])
else:  # If the dataframe is not empty then this code appends the row (dictionary) created from the applicant object's method to the end of the dataframe
    info_df.loc[len(info_df)] = applicant.app_info_to_dict()

8) Finally, save the two dataframes to csv files

### Happy coding!